In [19]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing import TypedDict, Dict, List
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: List[BaseMessage]
    summary: str


llm = ChatOllama(model="llama3", temprature=0.7)

def node_summary(state: State):
    
    print("Entering in summary \n\n\n")
    
    oldMessages = state["messages"][:4]
    summary = state["summary"]

    human = HumanMessage(content= f"""   
    Please update the summary
    old summary: {summary}
    new conversation: {oldMessages}
    """)

    system = SystemMessage(content= "You are a helpful summary expert which provides summary for given input")

    response = llm.invoke([system, human])

    state["messages"] = state["messages"][4:]
    state["summary"] = response.content

    print(f"summary is: {response.content} \n\n")

    return state


def node_llm(state: State):

    messages = []

    if state.get("summary"):
        messages.append(
            SystemMessage(content= f"""
                you have this conversation summary. Please answer the question based on this summary
                summary: {state["summary"]}
            """)
        )

    messages.extend(state["messages"])
        

    response = llm.invoke(messages)

    state["messages"].append(AIMessage(content=response.content))

    print(response.content)
    print("*****************************************************************")

    return state
    

def node_ask_question(state: State):
    
    print("\n\n\n")
    
    user = input("User:")

    print("\n\n")

    state["messages"].append(HumanMessage(content=user))

    return state


def router(state:State):
    last = state["messages"][-1].content.lower()
    if "exit" in last:
        return "end"
    elif len(state["messages"]) > 4:
         return "summary"
    
    return "llm"


graph = StateGraph(state_schema=State)

graph.add_node("ask_question", node_ask_question)
graph.add_node("llm", node_llm)
graph.add_node("summary", node_summary)

graph.add_edge(START, "ask_question")
graph.add_edge("llm", "ask_question")
graph.add_edge("summary", "llm")

graph.add_conditional_edges("ask_question", router, {"llm": "llm", "end": END, "summary": "summary"})

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

state = State(messages=[], summary="")

config = {
    "configurable": {
        "thread_id": "1"
    }
}
state = app.invoke(state, config=config)


User: hello





Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?
*****************************************************************






User: i am ashish and learning AI





Nice to meet you, Ashish!

That's awesome that you're interested in learning about Artificial Intelligence (AI)! AI is a fascinating field that has the potential to transform many aspects of our lives.

What specifically are you hoping to learn about AI? Are you looking to develop skills in machine learning, natural language processing, computer vision, or something else?

Feel free to share your goals and interests, and I'll do my best to help guide you on your AI-learning journey!
*****************************************************************






User: i want to be an AI engineer





Entering in summary 



summary is: Here is the updated summary:

The conversation starts with a human saying "hello". The AI responds with a friendly message, asking if there's something it can help with or if the human just wants to chat. The human introduces themselves as Ashish and mentions they are learning about AI. The AI welcomes Ashish and expresses excitement that they're interested in AI. It then asks Ashish what specific aspects of AI they'd like to learn about, such as machine learning, natural language processing, or computer vision. 


Exciting goal! According to the conversation summary, I should ask you:

"What specific aspects of AI engineering would you like to focus on, such as machine learning, natural language processing, or computer vision? Or are there any particular applications or industries that interest you?"
*****************************************************************






User: exit


In [42]:
history = app.get_state_history(config)

In [43]:
history

<generator object Pregel.get_state_history at 0x168bd5fe0>

In [44]:
list(history)[0]

StateSnapshot(values={'messages': [HumanMessage(content='i want to be an AI engineer', additional_kwargs={}, response_metadata={}), AIMessage(content='Exciting goal! According to the conversation summary, I should ask you:\n\n"What specific aspects of AI engineering would you like to focus on, such as machine learning, natural language processing, or computer vision? Or are there any particular applications or industries that interest you?"', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='exit', additional_kwargs={}, response_metadata={})], 'summary': 'Here is the updated summary:\n\nThe conversation starts with a human saying "hello". The AI responds with a friendly message, asking if there\'s something it can help with or if the human just wants to chat. The human introduces themselves as Ashish and mentions they are learning about AI. The AI welcomes Ashish and expresses excitement that they\'re interested in AI. It then asks